In [21]:
import torch
import numpy as np

x = torch.FloatTensor(np.array([[1 , 2 , 4],[3 , 4 , 5]])) #It is always better to use 32 Bits then the 64 as It takes half the computation and Speed
x

tensor([[1., 2., 4.],
        [3., 4., 5.]])

In [22]:
x[: , 1] = -1

In [23]:
x.relu_() #In place Operation

tensor([[1., 0., 4.],
        [3., 0., 5.]])

In [24]:
device = "cuda"
x = x.to(device=device)
x.device

device(type='cuda', index=0)

In [25]:
#Putting
learning_rate = 0.1
x = torch.tensor(5.0 , requires_grad=True)
for iteration in range(100):
    f = x**2
    f.backward()
    with torch.no_grad():
        x -=  learning_rate * x.grad
    x.grad.zero_()

In [26]:
#Implementing Linear Regression using Pytorch
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import torch

housing = fetch_california_housing()
x = housing.data
y = housing.target

In [27]:
x_train , x_test , y_train , y_test = train_test_split(x , y , random_state=42)
x_train , x_valid , y_train , y_valid = train_test_split(x_train , y_train , random_state=42)

In [28]:
x_train = torch.FloatTensor(x_train)
x_valid = torch.FloatTensor(x_valid)
x_test = torch.FloatTensor(x_test)

In [29]:
means = x_train.mean(dim= 0 , keepdim=True)
std = x_train.std(dim=0 , keepdim=True)
x_train = (x_train - means) / std
x_test = (x_test - means ) / std
x_valid = (x_valid - means) / std

In [30]:
y_train = torch.FloatTensor(y_train.reshape(-1 , 1))
y_test = torch.FloatTensor(y_test.reshape(-1 , 1))
y_valid = torch.FloatTensor(y_valid.reshape(-1 , 1))

In [31]:
torch.manual_seed(42)
n_featurs = x_train.shape[1]
w = torch.randn((n_featurs , 1) , requires_grad=True) #Random weigth initiliation
b = torch.tensor(0.0 , requires_grad=True) #Bias Term

In [32]:
learning_rate = 0.4
epochs = 20
for iterations in range(epochs):
    y_pred = x_train @ w + b #Forward Pass 
    loss = ((y_pred - y_train)** 2).mean()
    loss.backward() #Backward Pass
    with torch.no_grad():
        b -= learning_rate * b.grad
        w -= learning_rate * w.grad
        b.grad.zero_()
        w.grad.zero_()
    print(f"Epochs {iterations + 1} / {epochs} , Loss {loss.item()}")

Epochs 1 / 20 , Loss 16.158456802368164
Epochs 2 / 20 , Loss 4.879360675811768
Epochs 3 / 20 , Loss 2.255225896835327
Epochs 4 / 20 , Loss 1.3307620286941528
Epochs 5 / 20 , Loss 0.9680696129798889
Epochs 6 / 20 , Loss 0.814268171787262
Epochs 7 / 20 , Loss 0.7417048811912537
Epochs 8 / 20 , Loss 0.7020705342292786
Epochs 9 / 20 , Loss 0.676592230796814
Epochs 10 / 20 , Loss 0.6577968001365662
Epochs 11 / 20 , Loss 0.6426153779029846
Epochs 12 / 20 , Loss 0.6297225952148438
Epochs 13 / 20 , Loss 0.6184943914413452
Epochs 14 / 20 , Loss 0.6085970997810364
Epochs 15 / 20 , Loss 0.5998218655586243
Epochs 16 / 20 , Loss 0.5920187830924988
Epochs 17 / 20 , Loss 0.5850692391395569
Epochs 18 / 20 , Loss 0.5788735151290894
Epochs 19 / 20 , Loss 0.5733454823493958
Epochs 20 / 20 , Loss 0.5684101581573486


In [33]:
#Prediction
x_new = x_test[:3]
with torch.no_grad():
    y_pred = x_new @ w + b

print(y_pred)

tensor([[0.8916],
        [1.6480],
        [2.6577]])


In [34]:
#Using Higher Level API
import torch.nn as nn

torch.manual_seed(42)
model = nn.Linear(in_features= n_featurs , out_features= 1)
model.weight

Parameter containing:
tensor([[ 0.2703,  0.2935, -0.0828,  0.3248, -0.0775,  0.0713, -0.1721,  0.2076]],
       requires_grad=True)

In [35]:
learning_rate = 0.4
optimizer = torch.optim.SGD(model.parameters() , lr=learning_rate)

loss = nn.MSELoss()

In [36]:
#Train_Our model 
def train_bgd(model , optimizer , x_train , y_train , criterion , n_epochs): #Batch GD
    for epochs in range(n_epochs):
        y_pred = model(x_train)
        loss = criterion(y_pred , y_train) #Criterion is often Refered as the Loss function
        loss.backward()
        optimizer.step() #This is to Update the weights 
        optimizer.zero_grad() #This is to make the Gradient zero
        print(f"Epoch {epochs + 1}/{n_epochs}, Loss: {loss.item()}")

In [37]:
torch.manual_seed(42)
model_1 = nn.Sequential(
    nn.Linear(n_featurs , 50), #First Hidden Layer
    nn.ReLU(),
    nn.Linear(50 , 40), #Second Hidden Layer
    nn.ReLU(),
    nn.Linear(40 , 1) #Third Hidden Layer
)

learning_rate = 0.4
optimizer = torch.optim.SGD(model.parameters() , lr=learning_rate)
mse = nn.MSELoss()
train_bgd(model , optimizer , x_train , y_train  , loss ,epochs)

Epoch 1/20, Loss: 4.3378496170043945
Epoch 2/20, Loss: 0.7802932858467102
Epoch 3/20, Loss: 0.6253840327262878
Epoch 4/20, Loss: 0.6060433983802795
Epoch 5/20, Loss: 0.595629870891571
Epoch 6/20, Loss: 0.5873566269874573
Epoch 7/20, Loss: 0.5802990198135376
Epoch 8/20, Loss: 0.5741382241249084
Epoch 9/20, Loss: 0.5687101483345032
Epoch 10/20, Loss: 0.5639079213142395
Epoch 11/20, Loss: 0.5596510767936707
Epoch 12/20, Loss: 0.555873692035675
Epoch 13/20, Loss: 0.5525193810462952
Epoch 14/20, Loss: 0.5495391488075256
Epoch 15/20, Loss: 0.5468899011611938
Epoch 16/20, Loss: 0.5445338487625122
Epoch 17/20, Loss: 0.5424376130104065
Epoch 18/20, Loss: 0.5405715703964233
Epoch 19/20, Loss: 0.5389096736907959
Epoch 20/20, Loss: 0.5374288558959961


In [ ]:
from torch.utils.data import TensorDataset , DataLoader


tensor_dataset = TensorDataset(x_train , y_train)
train_loader = DataLoader(tensor_dataset , batch_size=32 , shuffle=True)
torch.manual_seed(42)
model = nn.Sequential(
    nn.Linear(n_featurs , 50),
    nn.ReLU(),
    nn.Linear(50 , 40),
    nn.ReLU(),
    nn.Linear(40 , 1)
)
model = model.to(device="cuda")

#Mini Batch Gradients
def train(model , optimizer , criterion , train_loader , n_epochs):
    model.train()
    for epochs in range(n_epochs):
        for x_batch, y_batch in train_loader:
            x_batch , y_batch = x_batch.to(device) , y_batch.to(device)
            y_pred = model(x_batch)
            loss = criterion(y_pred , y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            mean_loss = total_loss / len(train_loader)
            print(f"Epoch {epochs +1} / {n_epochs} , loss:{mean_loss:.4f}")

train(model , opti)